# Reality2Unity — Week 1 Baseline (Colab)

Google Colab에서 **런타임 → 런타임 유형 변경 → GPU**를 선택한 뒤 위에서부터 실행하세요.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/mir0173/Reality2Unity.git"
BRANCH = "develop"
REPO_DIR = Path("/content/Reality2Unity")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

required = [
    "python/main.py",
    "python/depth_estimation/midas.py",
    "python/terrain_generation/legacy_converter.py",
]


## 실행 패키지 준비



In [ ]:
%pip -q install timm opencv-python-headless matplotlib


In [ ]:
%matplotlib inline
import sys

python_dir = str(REPO_DIR / "python")
if python_dir not in sys.path:
    sys.path.insert(0, python_dir)

from main import main


## 입력 이미지 업로드



In [ ]:
from google.colab import files

uploaded = files.upload()
filename, image_bytes = next(iter(uploaded.items()))
input_dir = Path("/content/baseline_inputs")
input_dir.mkdir(parents=True, exist_ok=True)
image_path = input_dir / Path(filename).name
image_path.write_bytes(image_bytes)
del uploaded, image_bytes


## Baseline 실행



In [ ]:
main(str(image_path))


## 저장 전 출력 정리

실행 후 Git에 반영하기 전에는 Colab의 **수정 → 모든 출력 지우기**를 실행하세요. Colab 설정에서 **저장 시 코드 셀 출력 생략**을 활성화하세요.

# Reality2Unity — Week 2 (Colab)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from depth_estimation.midas import load_midas_model, estimate_depth

midas, transform, device = load_midas_model()

depth = estimate_depth(str(image_path), midas, transform, device)

image = cv2.imread(str(image_path))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print("Image shape:", image.shape)
print("Depth shape:", depth.shape)

H, W = depth.shape

depth_norm = (depth - depth.min()) / (depth.max() - depth.min())
Z = 1 / (depth_norm + 0.15)

fx = W
fy = W
cx = (W - 1) / 2
cy = (H - 1) / 2

step = 2

u, v = np.meshgrid(
    np.arange(0, W, step),
    np.arange(0, H, step)
)

z = Z[::step, ::step]
x = (u - cx) * z / fx
y = -(v - cy) * z / fy

colors = image[::step, ::step].reshape(-1, 3) / 255

mask = depth_norm[::step, ::step] > 0.1

rgb = image[::step, ::step] / 255.0

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(
    x[mask],
    y[mask],
    z[mask],
    c=rgb[mask],
    s=4
)

ax.view_init(elev=90, azim=-90)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('MiDaS 3D Point Cloud')

plt.show()